# Chapter 7: Di Build Chat Applications
## Microsoft Foundry Models API Quickstart

Dis notebook e base on top di [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) wey get notebooks wey dey access [Azure OpenAI](notebook-azure-openai.ipynb) services.

> **Note:** GitHub Models wan retire for end of July 2026. Dis notebook now dey target [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), wey get di same free-to-try model catalog and Azure AI Inference SDK experience.


# Overview  
"Big language models na functions wey dey map text go text. If you give am input text string, big language model dey try predict di text wey go follow"(1). Dis "quickstart" notebook go introduce users to high-level LLM concepts, main package requirements wey you need to start with AML, small introduction to prompt design, plus small examples of different use cases. 


## Table of Contents  

[Overview](#overview)  
[How to use OpenAI Service](#how-to-use-openai-service)  
[1. Creating your OpenAI Service](#1.-creating-your-openai-service)  
[2. Installation](#2.-installation)    
[3. Credentials](#3.-credentials)  

[Use Cases](#use-cases)    
[1. Summarize Text](#1.-summarize-text)  
[2. Classify Text](#2.-classify-text)  
[3. Generate New Product Names](#3.-generate-new-product-names)  
[4. Fine Tune a Classifier](#4.fine-tune-a-classifier)  

[References](#references)


### Build your first prompt  
Dis short exercise go give basic introduction for how to submit prompts to model for Microsoft Foundry Models for simple task "summarization".  


**Steps**:  
1. Install `azure-ai-inference` library for your python environment, if you never do am before.  
2. Load standard helper libraries and set up your Microsoft Foundry Models credentials.  
3. Choose model wey you go use for your task  
4. Create simple prompt for the model  
5. Submit your request to the model API!  


### 1. Install `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Import helper libraries and instantiate credentials


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Finding di correct model  
Models like GPT-4o and GPT-4o mini fit sabi and generate natural language, and dem dey for Microsoft Foundry Models katalog beside models from Meta, Mistral, Cohere, and Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. Prompt Design  

"Di magic wey dey big language models be say by training dem to reduce dis prediction error for plenty text, di models go learn concepts wey dey useful for dis kind prediction. For example, dem go sabi concepts like"(1):

* how to spell
* how grammar dey work
* how to paraphrase
* how to answer questions
* how to hold conversation
* how to write for many languages
* how to code
* etc.

#### How to control big language model  
"Among all di inputs wey big language model get, di text prompt na di one wey get di most power(1).

Big language models fit use few ways to prompt to make output:

Instruction: Tell di model wetin you want
Completion: Make di model complete di beginning part of wetin you want
Demonstration: Show di model wetin you want, with either:
Some examples for di prompt
Plenty hundreds or thousands examples for fine-tuning training dataset"



#### Dem get three basic rules to create prompts:

**Show and tell**. Make am clear wetin you wan either through instructions, examples, or mix of both. If you wan make di model rank list of things for alphabetical order or classify paragraph by sentiment, show am say na wetin you want.

**Provide quality data**. If you dey try build classifier or make di model follow pattern, make sure say examples plenty. Make sure say you check your examples well — di model sabi notice normal spelling mistakes and fit still give you answer, but e fit think say e no be mistake and e fit affect di answer.

**Check your settings.** Di temperature and top_p settings dey control how sure di model dey when e dey generate answer. If you dey ask for answer wey get only one correct reply, you go wan set dem lower. If you dey find many different answers, you fit set dem higher. Di biggest mistake wey people dey make with these settings na to think say na "cleverness" or "creativity" controls dey be that thing.


Source: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Submit!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Call am same one again, how dem result dem compare?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Summarize Text  
#### Challenge  
Summarize text by adding a 'tl;dr:' to the end of a text passage. Notice how de model sabi how to do plenty tasks without extra instructions. You fit try more descriptive prompts pass tl;dr to change how model go do and make the summary wey you go get special(3).  

Recent work don show better results for many NLP tasks and benchmarks by pre-training on big set of text then fine-tuning on one task specifically. Even though dem no design the architecture for particular task, still e need fine-tuning for each task with thousands or tens of thousands examples. But nigeria people fit do new language task with just small examples or simple instructions - na wetin NLP system no too fit do till now. For here, we show say if you make language models big pass, e go better for task-agnostic, few-shot performance, sometimes e fit even compete with prior best fine-tuning ways. 



Tl;dr


# Exercises for plenti use case  
1. Summarize Text  
2. Classify Text  
3. Generate New Product Names


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Classify Text  
#### Challenge  
Classify items into categories wey dem give for inference time. For di example wey dey below, we give both di categories and di text wey we go classify for di prompt(*playground_reference). 

Customer Inquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:

Classified category:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generate New Product Names
#### Challenge
Create product names from examples words. Here we include in the prompt information about the product we are going to generate names for. We also provide a similar example to show the pattern we wish to receive. We have also set the temperature value high to increase randomness and more innovative responses.

Product description: A home milkshake maker
Seed words: fast, healthy, compact.
Product names: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Product description: A pair of shoes wey fit any foot size.
Seed words: adaptable, fit, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# References  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Best practices for fine-tuning GPT models to classify text](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# For Mo Help  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# People Wey Contribute
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dis document don translate wit AI translation service [Co-op Translator](https://github.com/Azure/co-op-translator). Even tho we dey try make am correct, abeg make you know say automated translation fit get errors or mistakes. Di original document for dia own language na im be di correct source. For important info, make person wey sabi human translation do am. We no go responsible for any misunderstanding or wrong understanding wey fit happen because of dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
